# Mastodon — Spark Structured Streaming

Ce notebook lit les publications Mastodon depuis Kafka, nettoie et filtre
les données, enregistre les publications dans PostgreSQL et calcule des statistiques en streaming.

Architecture : API Mastodon → Producteur Python → Kafka → Spark → PostgreSQL.

Le producteur interroge le fil du hashtag #AI toutes les 10 secondes.
Les horodatages de publication et les fenêtres d’analyse utilisent le fuseau UTC.
Exécutez les sections 1 à 8 dans l’ordre. Exécutez la section 9 uniquement pour arrêter la démonstration.

L’aperçu s’exécute de manière asynchrone : attendez des publications avant de l’arrêter
dans la cellule suivante.

N’exécutez pas simultanément le script Spark de streaming séparé et ce notebook :
ils partagent le checkpoint pour PostgreSQL.

Conservez les dossiers de checkpoints et les volumes Kafka/PostgreSQL lors du redémarrage.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("MastodonStreamingNotebook")
    .master("local[2]")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.jars.ivy", "/tmp/spark-ivy")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3,"
        "org.postgresql:postgresql:42.7.4",
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")


In [4]:
print("Version de Spark :", spark.version)
spark.range(5).show()

Version de Spark : 3.5.3
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



## 1. Ingestion depuis Kafka

Spark lit les messages JSON du topic `mastodon_stream`.
Un schéma explicite définit les champs attendus.

Lors de la première exécution, `earliest` lit l’historique disponible dans Kafka.
Lors des exécutions suivantes, le checkpoint détermine où le traitement reprend.


In [5]:
schema = T.StructType([
    T.StructField("toot_id", T.StringType()),
    T.StructField("created_at", T.StringType()),
    T.StructField("user_id", T.StringType()),
    T.StructField("username", T.StringType()),
    T.StructField("content", T.StringType()),
    T.StructField("language", T.StringType()),
    T.StructField("hashtags", T.ArrayType(T.StringType())),
    T.StructField("favourites_count", T.IntegerType()),
    T.StructField("reblogs_count", T.IntegerType()),
])

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:29092")
    .option("subscribe", "mastodon_stream")
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", "1000")
    .load()
)

parsed_stream = (
    raw_stream
    .select(
        F.from_json(
            F.col("value").cast("string"),
            schema,
        ).alias("post")
    )
    .select("post.*")
)

print("DataFrame en streaming :", parsed_stream.isStreaming)
parsed_stream.printSchema()

DataFrame en streaming : True
root
 |-- toot_id: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- username: string (nullable = true)
 |-- content: string (nullable = true)
 |-- language: string (nullable = true)
 |-- hashtags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- favourites_count: integer (nullable = true)
 |-- reblogs_count: integer (nullable = true)



## 2. Nettoyage et filtrage par langue

Le contenu HTML est converti en texte brut et les dates de publication sont converties
en horodatages. Les enregistrements avec des identifiants manquants, des dates invalides
ou un contenu vide sont exclus.

Toutes les publications valides restent disponibles pour être enregistrées.
Les publications en anglais sont sélectionnées séparément pour les statistiques en streaming,
afin d’utiliser une même langue pour l’analyse.


In [6]:
from html.parser import HTMLParser


class TextExtractor(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.parts = []

    def handle_data(self, data):
        self.parts.append(data)

    def handle_starttag(self, tag, attrs):
        if tag in ("p", "br", "div"):
            self.parts.append(" ")

    def handle_endtag(self, tag):
        if tag in ("p", "div"):
            self.parts.append(" ")


@F.udf(returnType=T.StringType())
def clean_html(content):
    if content is None:
        return None

    parser = TextExtractor()
    parser.feed(content)
    parser.close()

    return " ".join("".join(parser.parts).split())


cleaned_stream = (
    parsed_stream
    .withColumn("created_at", F.to_timestamp("created_at"))
    .withColumn("content", clean_html(F.col("content")))
    .withColumn("language", F.lower(F.trim(F.col("language"))))
    .filter(
        F.col("toot_id").isNotNull()
        & (F.length(F.trim(F.col("toot_id"))) > 0)
        & F.col("user_id").isNotNull()
        & F.col("created_at").isNotNull()
        & (F.length("content") > 0)
    )
    .withColumn("text_length", F.length("content"))
)

english_stream = cleaned_stream.filter(
    F.col("language") == "en"
)

print("Flux nettoyé prêt.")
print("Flux en anglais prêt pour les statistiques.")

Flux nettoyé prêt.
Flux en anglais prêt pour les statistiques.


## 3. Aperçu

Une requête temporaire affiche dans la console les publications en anglais nettoyées.
Elle est arrêtée avant de lancer les requêtes d’écriture dans PostgreSQL.


In [7]:
from pathlib import Path

PROJECT_ROOT = Path("/opt/spark/work-dir")
CHECKPOINT_ROOT = PROJECT_ROOT / "checkpoints"

# Éviter de lancer deux aperçus en réexécutant cette cellule.
if "preview_query" in globals() and preview_query.isActif :
    preview_query.stop()

preview_query = (
    english_stream
    .select(
        "toot_id",
        "created_at",
        "username",
        "language",
        F.substring("content", 1, 120).alias("content_preview"),
        "text_length",
    )
    .writeStream
    .queryName("english_preview")
    .format("console")
    .outputMode("append")
    .option("truncate", "false")
    .option("numRows", "5")
    .option(
        "checkpointLocation",
        str(CHECKPOINT_ROOT / "notebook_preview"),
    )
    .trigger(processingTime="10 seconds")
    .start()
)

print("Aperçu démarré :", preview_query.isActive)

Aperçu démarré : True


In [8]:
preview_query.stop()
print("Aperçu arrêté.")

Aperçu arrêté.


26/10/08 16:26:56 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 4, writer: ConsoleWriter[numRows=5, truncate=false]] is aborting.
26/10/08 16:26:56 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 4, writer: ConsoleWriter[numRows=5, truncate=false]] aborted.
26/10/08 16:26:57 ERROR Utils: Aborting task                        (0 + 1) / 3]
org.apache.spark.TaskKilledException
	at org.apache.spark.TaskContextImpl.killTaskIfInterrupted(TaskContextImpl.scala:267)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:36)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:491)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage3.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(Bu

## 4. Enregistrement des publications nettoyées dans PostgreSQL

Chaque micro-batch est écrit dans une transaction PostgreSQL.
La contrainte d’unicité sur `toot_id` empêche les doublons de publications.

Les publications valides de toutes les langues sont enregistrées pour l’analyse historique.
Le flux contenant uniquement les publications en anglais sera utilisé pour les statistiques en streaming.

Si un batch échoue, l’erreur est transmise à Spark.
Lors d’une relecture, les insertions peuvent être retentées sans créer de doublons grâce à ON CONFLICT DO NOTHING.


In [9]:
import psycopg2


def write_posts(batch_df, batch_id):
    connection = psycopg2.connect(connect_timeout=10)

    processed = 0
    inserted = 0

    sql = """
        INSERT INTO toots (
            toot_id, created_at, user_id, username,
            content, language, hashtags,
            favourites_count, reblogs_count
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (toot_id) DO NOTHING
    """

    try:
        with connection:
            with connection.cursor() as cursor:
                rows = (
                    batch_df
                    .select(
                        "toot_id",
                        F.date_format(
                            "created_at",
                            "yyyy-MM-dd HH:mm:ss.SSSSSS",
                        ).alias("created_at"),
                        "user_id",
                        "username",
                        "content",
                        "language",
                        "hashtags",
                        "favourites_count",
                        "reblogs_count",
                    )
                    .toLocalIterator()
                )

                for row in rows:
                    cursor.execute(
                        sql,
                        (
                            row.toot_id,
                            row.created_at,
                            row.user_id,
                            row.username,
                            row.content,
                            row.language,
                            row.hashtags or [],
                            row.favourites_count or 0,
                            row.reblogs_count or 0,
                        ),
                    )

                    processed += 1
                    inserted += cursor.rowcount

        print(
            f"Micro-batch {batch_id} : "
            f"{inserted} publications insérées, "
            f"{processed - inserted} doublons ignorés.",
            flush=True,
        )

    finally:
        connection.close()

In [10]:
if "posts_query" in globals() and posts_query.isActif :
    posts_query.stop()

posts_query = (
    cleaned_stream.writeStream
    .queryName("postgres_posts")
    .foreachBatch(write_posts)
    .outputMode("append")
    .option(
        "checkpointLocation",
        str(CHECKPOINT_ROOT / "postgres"),
    )
    .trigger(processingTime="10 seconds")
    .start()
)

print("Traitement d’écriture dans PostgreSQL actif :", posts_query.isActive)

Traitement d’écriture dans PostgreSQL actif : True


Micro-batch 213 : 91 publications insérées, 0 doublons ignorés.


## 5. Vérification des données enregistrées

Les requêtes suivantes vérifient le nombre total de publications enregistrées,
affichent les enregistrements récents et vérifient l’unicité des identifiants des publications.
Ces vérifications peuvent être réexécutées pendant que le traitement d’écriture en streaming est actif.


In [11]:
import pandas as pd


def read_postgres(sql):
    connection = psycopg2.connect(connect_timeout=10)

    try:
        with connection.cursor() as cursor:
            cursor.execute(sql)

            columns = [
                description[0]
                for description in cursor.description
            ]

            return pd.DataFrame(
                cursor.fetchall(),
                columns=columns,
            )
    finally:
        connection.close()


display(read_postgres("""
    SELECT
        COUNT(*) AS total_posts,
        COUNT(DISTINCT toot_id) AS distinct_posts,
        MAX(created_at) AS latest_publication_utc
    FROM toots
"""))

display(read_postgres("""
    SELECT
        toot_id,
        created_at AS publication_utc,
        username,
        language,
        LEFT(content, 120) AS content_preview
    FROM toots
    ORDER BY created_at DESC
    LIMIT 5
"""))

,total_posts,distinct_posts,latest_publication_utc
0,1725,1725,2026-10-08 16:25:04.899


,toot_id,publication_utc,username,language,content_preview
0,117406217331907014,2026-10-08 16:25:04.899,cyberowy,pl,📊 Gemini 4 Argon: AI Google do łamania i łatan...
1,117406217338178908,2026-10-08 16:25:03.000,fritzlabsX@mas.to,en,"""NeuroAssistent"" Neuroni provides accessible a..."
2,117406214646464323,2026-10-08 16:24:23.918,mergedcode,en,Today's Tech News — top stories for builders. ...
3,117406211082516821,2026-10-08 16:23:19.000,techwire@gamefan.net,en,Samsung reportedly can’t make a profit on its ...
4,117406210048345029,2026-10-08 16:23:10.000,feed@igeek.gamer-geek-news.com,en,"🎮 All hail the long steam machine, Pimax's new..."


## 6. Publications en anglais par heure

Les publications sont regroupées en fenêtres d’une heure selon leur date de publication
(temps de l’événement), en UTC.

Les identifiants distincts des publications sont comptés pour éviter de compter deux fois les publications relues.
Un watermark d’un jour limite l’état conservé. Les publications arrivant avec plus
d’un jour de retard par rapport au temps de l’événement le plus récent observé peuvent être exclues.

Les comptages sont provisoires tant qu’une fenêtre reste ouverte.


In [12]:
connection = psycopg2.connect(connect_timeout=10)

try:
    with connection:
        with connection.cursor() as cursor:
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS streaming_posts_per_hour (
                    window_start TIMESTAMP PRIMARY KEY,
                    window_end TIMESTAMP NOT NULL,
                    post_count BIGINT NOT NULL,
                    updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
                )
            """)
finally:
    connection.close()

print("Table des statistiques horaires prête.")

Table des statistiques horaires prête.


In [13]:
hour_stats = (
    english_stream
    .withWatermark("created_at", "1 day")
    .groupBy(
        F.window("created_at", "1 hour")
    )
    .agg(
        F.size(
            F.collect_set("toot_id")
        ).cast("long").alias("post_count")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "post_count",
    )
)

hour_stats.printSchema()

root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- post_count: long (nullable = false)



In [14]:
def write_hour_stats(batch_df, batch_id):
    connection = psycopg2.connect(connect_timeout=10)
    updated = 0

    sql = """
        INSERT INTO streaming_posts_per_hour (
            window_start, window_end, post_count
        )
        VALUES (%s, %s, %s)

        ON CONFLICT (window_start)
        DO UPDATE SET
            window_end = EXCLUDED.window_end,
            post_count = EXCLUDED.post_count,
            updated_at = NOW()
    """

    try:
        with connection:
            with connection.cursor() as cursor:
                rows = (
                    batch_df
                    .select(
                        F.date_format(
                            "window_start",
                            "yyyy-MM-dd HH:mm:ss",
                        ).alias("window_start"),
                        F.date_format(
                            "window_end",
                            "yyyy-MM-dd HH:mm:ss",
                        ).alias("window_end"),
                        "post_count",
                    )
                    .toLocalIterator()
                )

                for row in rows:
                    cursor.execute(
                        sql,
                        (
                            row.window_start,
                            row.window_end,
                            row.post_count,
                        ),
                    )
                    updated += 1

        print(
            f"Micro-batch de statistiques horaires {batch_id} : "
            f"{updated} fenêtres mises à jour.",
            flush=True,
        )

    finally:
        connection.close()

In [15]:
# Éviter de lancer deux requêtes horaires.
if "hour_query" in globals() and hour_query.isActif :
    hour_query.stop()

hour_query = (
    hour_stats.writeStream
    .queryName("english_posts_per_hour")
    .foreachBatch(write_hour_stats)
    .outputMode("update")
    .option(
        "checkpointLocation",
        str(CHECKPOINT_ROOT / "english_posts_per_hour"),
    )
    .trigger(processingTime="10 seconds")
    .start()
)

print("Statistiques horaires actives :", hour_query.isActive)

Statistiques horaires actives : True


In [16]:
spark.sparkContext.setLogLevel("ERROR")

print("Actif :", hour_query.isActive)
print("Erreur :", hour_query.exception())

display(read_postgres("""
    SELECT
        window_start AS hour_utc,
        window_end,
        post_count
    FROM streaming_posts_per_hour
    ORDER BY window_start DESC
    LIMIT 10
"""))

Actif : True
Erreur : None


,hour_utc,window_end,post_count
0,2026-10-08 11:00:00,2026-10-08 12:00:00,14
1,2026-10-08 10:00:00,2026-10-08 11:00:00,54
2,2026-10-08 09:00:00,2026-10-08 10:00:00,60
3,2026-10-08 08:00:00,2026-10-08 09:00:00,62
4,2026-10-08 07:00:00,2026-10-08 08:00:00,48
5,2026-10-08 06:00:00,2026-10-08 07:00:00,38
6,2026-10-08 05:00:00,2026-10-08 06:00:00,59
7,2026-10-08 04:00:00,2026-10-08 05:00:00,45
8,2026-10-08 03:00:00,2026-10-08 04:00:00,51
9,2026-10-08 02:00:00,2026-10-08 03:00:00,39


## 7. Longueur moyenne des publications par utilisateur et par heure

Les publications en anglais sont regroupées par utilisateur et par fenêtre d’une heure selon leur date de publication,
en utilisant des horodatages UTC.

La longueur moyenne est mesurée en caractères après le nettoyage du HTML.
Les publications identiques répétées sont comptées une seule fois.

Un watermark d’un jour limite l’état conservé.
Les résultats sont mis à jour dans PostgreSQL à chaque micro-batch et restent
provisoires tant que la fenêtre est ouverte.


In [49]:
connection = psycopg2.connect(connect_timeout=10)

try:
    with connection:
        with connection.cursor() as cursor:
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS streaming_user_stats_hourly (
                    window_start TIMESTAMP NOT NULL,
                    window_end TIMESTAMP NOT NULL,
                    user_id VARCHAR(100) NOT NULL,
                    post_count BIGINT NOT NULL,
                    avg_text_length DOUBLE PRECISION NOT NULL,
                    updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
                    PRIMARY KEY (window_start, user_id)
                )
            """)
finally:
    connection.close()

print("Table des statistiques horaires par utilisateur prête.")

Hourly user statistics table ready.


In [50]:
user_stats_hourly = (
    english_stream
    .withWatermark("created_at", "1 day")
    .groupBy(
        F.window("created_at", "1 hour"),
        "user_id",
    )
    .agg(
        F.collect_set(
            F.struct("toot_id", "text_length")
        ).alias("posts")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "user_id",
        F.size("posts").cast("long").alias("post_count"),
        F.expr("""
            aggregate(
                posts,
                CAST(0 AS BIGINT),
                (total, post) -> total + post.text_length
            ) / size(posts)
        """).cast("double").alias("avg_text_length"),
    )
)

user_stats_hourly.printSchema()

root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- post_count: long (nullable = false)
 |-- avg_text_length: double (nullable = true)



In [51]:
def write_user_stats_hourly(batch_df, batch_id):
    connection = psycopg2.connect(connect_timeout=10)
    updated = 0

    sql = """
        INSERT INTO streaming_user_stats_hourly (
            window_start,
            window_end,
            user_id,
            post_count,
            avg_text_length
        )
        VALUES (%s, %s, %s, %s, %s)

        ON CONFLICT (window_start, user_id)
        DO UPDATE SET
            window_end = EXCLUDED.window_end,
            post_count = EXCLUDED.post_count,
            avg_text_length = EXCLUDED.avg_text_length,
            updated_at = NOW()
    """

    try:
        with connection:
            with connection.cursor() as cursor:
                rows = (
                    batch_df
                    .select(
                        F.date_format(
                            "window_start",
                            "yyyy-MM-dd HH:mm:ss",
                        ).alias("window_start"),
                        F.date_format(
                            "window_end",
                            "yyyy-MM-dd HH:mm:ss",
                        ).alias("window_end"),
                        "user_id",
                        "post_count",
                        "avg_text_length",
                    )
                    .toLocalIterator()
                )

                for row in rows:
                    cursor.execute(
                        sql,
                        (
                            row.window_start,
                            row.window_end,
                            row.user_id,
                            row.post_count,
                            row.avg_text_length,
                        ),
                    )
                    updated += 1

        print(
            f"Micro-batch de statistiques horaires par utilisateur {batch_id} : "
            f"{updated} lignes mises à jour.",
            flush=True,
        )

    finally:
        connection.close()

In [53]:
# Éviter de lancer deux requêtes horaires.
if "user_query_hourly" in globals() and user_query_hourly.isActif :
    user_query_hourly.stop()

user_query_hourly = (
    user_stats_hourly.writeStream
    .queryName("english_user_statistics_hourly")
    .foreachBatch(write_user_stats_hourly)
    .outputMode("update")
    .option(
        "checkpointLocation",
        str(CHECKPOINT_ROOT / "english_user_statistics_hourly"),
    )
    .trigger(processingTime="10 seconds")
    .start()
)

In [54]:
print("Actif :", user_query_hourly.isActive)
print("Erreur :", user_query_hourly.exception())

display(read_postgres("""
    SELECT
        window_start AS hour_utc,
        user_id,
        post_count,
        ROUND(avg_text_length::numeric, 1) AS avg_characters
    FROM streaming_user_stats_hourly
    ORDER BY window_start DESC, post_count DESC, user_id
    LIMIT 10
"""))

Active: True
Error: None


,hour_utc,user_id,post_count,avg_characters
0,2026-10-08 09:00:00,116070447239053927,8,454.3
1,2026-10-08 09:00:00,115061140460610278,4,7432.0
2,2026-10-08 09:00:00,116450756112437762,4,387.3
3,2026-10-08 09:00:00,116988275448638305,3,332.3
4,2026-10-08 09:00:00,112026408859289825,2,335.0
5,2026-10-08 09:00:00,113373281572068351,2,304.5
6,2026-10-08 09:00:00,116467552435501973,2,467.5
7,2026-10-08 09:00:00,107320075805792927,1,220.0
8,2026-10-08 09:00:00,109212695704203107,1,360.0
9,2026-10-08 09:00:00,109318490906315885,1,635.0


## 8. Surveillance des traitements en streaming

Vérifiez que chaque requête de streaming est active et examinez les éventuelles erreurs.
Chaque requête traite les nouvelles données Kafka toutes les 10 secondes.


In [55]:
queries = {
    "Publications → PostgreSQL": posts_query,
    "Publications par heure": hour_query,
    "Statistiques par utilisateur et par heure": user_query_hourly,
}

for name, query in queries.items():
    print(f"\n{name}")
    print("Actif :", query.isActive)
    print("Erreur :", query.exception())
    print("Statut :", query.status["message"])


Posts → PostgreSQL
Active: True
Error: None
Status: Waiting for next trigger

Posts per hour
Active: True
Error: None
Status: Waiting for next trigger

User statistics per hour
Active: True
Error: None
Status: Waiting for next trigger


## 9. Arrêt des requêtes de streaming

Exécutez la cellule suivante uniquement lorsque la démonstration est terminée.
Les checkpoints sont conservés afin de reprendre le traitement au redémarrage.


In [ ]:
for query in list(spark.streams.active):
    query.stop()
    print(f"Arrêté : {query.name}")

print("Nombre de requêtes encore actives :", len(spark.streams.active))

Pour reprendre dans le même noyau, réexécutez les cellules de démarrage de posts_query,
hour_query et user_query_hourly.

Après le redémarrage du noyau, réexécutez le notebook jusqu’à la section 8 incluse.
